# TabPFN v3 and v2.5

For instructions on how to install, dependencies and use TabPFN, consult https://github.com/PriorLabs/tabpfn

In [ ]:
from pathlib import Path
from getpass import getpass
import os
import pickle
import time
import numpy as np
import pandas as pd
import torch
import tabpfn
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

SEED = 94
DEVICE = 'cuda'

## Prepare the three feature blocks

In [ ]:
project_path = Path('/home/boscoll/Projects/cdk_predict')
train_test_split = project_path / 'data/development_cohort/train_test_split/genomic_train_test_split_msk_final'
clinical_path = project_path / 'data/development_cohort/clinical/clinical_msk_final.csv'


with open(train_test_split, 'rb') as f:
    X_train_genomic, X_holdout_locked, y_train_surv, y_holdout_locked = pickle.load(f)
y_train_surv = y_train_surv.loc[X_train_genomic.index, ['Time', 'Event']].copy()

clinical = pd.read_csv(clinical_path, index_col=0)

#from the clinical only dataframe, remove the 'met_sample' variable
X_train_clinical = clinical.loc[X_train_genomic.index].copy()
X_train_clinical.drop(columns='met_sample', inplace=True)

X_train = pd.concat([clinical.loc[X_train_genomic.index], X_train_genomic], axis=1) 

feature_sets = {
    'clinical': X_train_clinical,
    'genomic': X_train_genomic,
    'clinicogenomic': X_train,
}

In [ ]:
#patients with FU time < 12 months and censored are considered unknown at 12 months
def twelve_month_target(y):
    event_by_12 = y['Event'].astype(bool) & y['Time'].le(12)
    known_at_12 = event_by_12 | y['Time'].ge(12)
    return event_by_12.astype('int8'), known_at_12

## Generate five-fold OOF risks

In [ ]:
if not os.environ.get('TABPFN_TOKEN'):
    token = getpass('token: ').strip()

    os.environ['TABPFN_TOKEN'] = token
    del token

torch.cuda.set_device(0)

model_versions = {
    'tabpfn_v3': ModelVersion.V3,
    'tabpfn_v2_5': ModelVersion.V2_5,
}
score_names = [
    f'{feature_block}__{model_name}'
    for feature_block in feature_sets
    for model_name in model_versions
]
oof_score = pd.DataFrame(np.nan, index=X_train.index, columns=score_names)
fold_rows = []

splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
y_12_all, _ = twelve_month_target(y_train_surv)
splits = splitter.split(X_train, y_12_all)

for fold, (train_position, validation_position) in enumerate(splits, start=1):
    train_index = X_train.index[train_position]
    validation_index = X_train.index[validation_position]

    #exclude those w/o event and follow up time < 12 months
    y_12, known_at_12 = twelve_month_target(y_train_surv.loc[train_index])
    eligible_train_index = train_index[known_at_12.to_numpy()]

    for feature_block, X_model in feature_sets.items():
        X_fold_train = X_model.loc[eligible_train_index]
        y_fold_train = y_12.loc[eligible_train_index]
        X_fold_validation = X_model.loc[validation_index]

        for model_name, version in model_versions.items():
            score_name = f'{feature_block}__{model_name}'
            model = TabPFNClassifier.create_default_for_version(
                version,
                device=DEVICE,
                n_estimators='auto',
                categorical_features_indices=list(range(X_fold_train.shape[1])),
                random_state=SEED + fold,
                balance_probabilities=False,
                show_progress_bar=False,
            )
            started = time.perf_counter()
            model.fit(X_fold_train, y_fold_train)
            oof_score.loc[validation_index, score_name] = model.predict_proba(
                X_fold_validation
            )[:, 1]
            fold_rows.append({
                'fold': fold,
                'model': score_name,
                'n_training_rows': len(X_fold_train),
                'seconds': time.perf_counter() - started,
            })
            print(f'Fold {fold}/5, {score_name}: {fold_rows[-1]["seconds"]:.1f}s')
            del model
            torch.cuda.empty_cache()

fold_audit = pd.DataFrame(fold_rows)

Using: NVIDIA A40
Fold 1/5, clinicogenomic__tabpfn_v3: 4.0s
Fold 1/5, clinicogenomic__tabpfn_v2_5: 5.2s
Fold 1/5, clinical__tabpfn_v3: 8.8s
Fold 1/5, clinical__tabpfn_v2_5: 3.2s
Fold 1/5, genomic__tabpfn_v3: 3.9s
Fold 1/5, genomic__tabpfn_v2_5: 3.9s
Fold 2/5, clinicogenomic__tabpfn_v3: 4.2s
Fold 2/5, clinicogenomic__tabpfn_v2_5: 4.4s
Fold 2/5, clinical__tabpfn_v3: 3.9s
Fold 2/5, clinical__tabpfn_v2_5: 3.3s
Fold 2/5, genomic__tabpfn_v3: 3.9s
Fold 2/5, genomic__tabpfn_v2_5: 3.8s
Fold 3/5, clinicogenomic__tabpfn_v3: 4.2s
Fold 3/5, clinicogenomic__tabpfn_v2_5: 4.5s
Fold 3/5, clinical__tabpfn_v3: 4.0s
Fold 3/5, clinical__tabpfn_v2_5: 3.4s
Fold 3/5, genomic__tabpfn_v3: 4.1s
Fold 3/5, genomic__tabpfn_v2_5: 4.0s
Fold 4/5, clinicogenomic__tabpfn_v3: 4.2s
Fold 4/5, clinicogenomic__tabpfn_v2_5: 4.3s
Fold 4/5, clinical__tabpfn_v3: 4.1s
Fold 4/5, clinical__tabpfn_v2_5: 3.0s
Fold 4/5, genomic__tabpfn_v3: 4.3s
Fold 4/5, genomic__tabpfn_v2_5: 3.9s
Fold 5/5, clinicogenomic__tabpfn_v3: 4.3s
Fold 5/5, cl

,fold,model,n_training_rows,seconds
0,1,clinicogenomic__tabpfn_v3,655,3.991934
1,1,clinicogenomic__tabpfn_v2_5,655,5.179723
2,1,clinical__tabpfn_v3,655,8.782255
3,1,clinical__tabpfn_v2_5,655,3.237797
4,1,genomic__tabpfn_v3,655,3.897820
5,1,genomic__tabpfn_v2_5,655,3.896756
6,2,clinicogenomic__tabpfn_v3,659,4.160188
7,2,clinicogenomic__tabpfn_v2_5,659,4.415600
8,2,clinical__tabpfn_v3,659,3.882053
9,2,clinical__tabpfn_v2_5,659,3.328325


## Save 12-month risks

The predicted probability is expressed on a 0–100 scale. The CSV contains only `record_id` and the six comparison-ready risks.

In [ ]:
# oof_risk = oof_score.mul(100)
# oof_risk.index.name = 'record_id'
# oof_risk.to_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabpfn_risks.csv')

# Cross-fitted feature-deletion analysis

In [ ]:
import gc

RELIANCE_MUTATIONS = [
    'BRCA1_tsg', 'BRCA2_tsg', 'PTEN_tsg', 'RB1_tsg', 'PALB2_tsg',
]
RELIANCE_MODEL_VERSIONS = {
    'TabPFN v3': ModelVersion.V3,
    'TabPFN v2.5': ModelVersion.V2_5,
}

# Pool the original development and locked-holdout cohorts.
X_genomic_all = pd.concat(
    [X_train_genomic, X_holdout_locked], axis=0
).replace([np.inf, -np.inf], np.nan).astype('float32')
X_genomic_all = X_genomic_all.loc[
    :, ~X_genomic_all.columns.str.contains('fga', case=False)
]
y_reliance = pd.concat([
    y_train_surv[['Time', 'Event']],
    y_holdout_locked[['Time', 'Event']],
], axis=0).loc[X_genomic_all.index]

# Match the clinicogenomic block above while excluding met_sample.
X_clinical_all = clin.drop(columns='met_sample').loc[X_genomic_all.index]
X_reliance = pd.concat(
    [X_clinical_all, X_genomic_all], axis=1
).astype('float32')

missing_genes = sorted(set(RELIANCE_MUTATIONS) - set(X_reliance.columns))

In [ ]:
reliance_rows = []

for gene_nr, gene in enumerate(tqdm(RELIANCE_MUTATIONS, desc='Alterations')):
    carrier_status = X_reliance[gene].astype('float32').astype(int)

    for repeat in tqdm(
        range(30), desc=gene, leave=False
    ):
        outer_cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=SEED + repeat,
        )

        for outer_fold, (train_pos, test_pos) in enumerate(
            outer_cv.split(X_reliance, carrier_status), start=1
        ):
            seed = SEED + gene_nr * 10_000 + repeat * 10 + outer_fold
            train_index = X_reliance.index[train_pos]
            test_index = X_reliance.index[test_pos]

            y_12, known_at_12 = twelve_month_target(
                y_reliance.loc[train_index]
            )
            eligible_train_index = train_index[known_at_12.to_numpy()]
            X_fold_train = X_reliance.loc[eligible_train_index]
            y_fold_train = y_12.loc[eligible_train_index]
            if y_fold_train.nunique() != 2:
                raise ValueError(
                    f'{gene}, repeat {repeat}, fold {outer_fold}: '
                    'the eligible training target does not contain both classes'
                )

            carrier_index = test_index[
                carrier_status.loc[test_index].to_numpy() == 1
            ]
            X_observed = X_reliance.loc[carrier_index].copy()
            X_ablated = X_observed.copy()
            X_ablated[gene] = 0

            for model_name, version in RELIANCE_MODEL_VERSIONS.items():
                clf = TabPFNClassifier.create_default_for_version(
                    version,
                    device=DEVICE,
                    n_estimators='auto',
                    categorical_features_indices=list(
                        range(X_fold_train.shape[1])
                    ),
                    random_state=seed,
                    balance_probabilities=False,
                    show_progress_bar=False,
                )
                clf.fit(X_fold_train, y_fold_train)
                positive_class_position = int(
                    np.flatnonzero(clf.classes_ == 1)[0]
                )
                risk_observed = (
                    clf.predict_proba(X_observed)[
                        :, positive_class_position
                    ] * 100
                )
                risk_ablated = (
                    clf.predict_proba(X_ablated)[
                        :, positive_class_position
                    ] * 100
                )

                for record_id, observed, ablated in zip(
                    carrier_index, risk_observed, risk_ablated
                ):
                    delta = observed - ablated
                    reliance_rows.append({
                        'record_id': record_id,
                        'gene': gene,
                        'model': model_name,
                        'repeat': repeat,
                        'outer_fold': outer_fold,
                        'risk_observed': observed,
                        'risk_ablated': ablated,
                        'delta': delta,
                        'absolute_delta': abs(delta),
                    })

                del clf
                gc.collect()
                torch.cuda.empty_cache()

reliance_raw = pd.DataFrame(reliance_rows)
# reliance_raw.to_csv(
#     '/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabpfn_model_reliance/reliance_all_repeats.csv', index=False)


In [ ]:
# Median over repeated OOF estimates for each patient.
reliance_patients = reliance_raw.groupby(
    ['record_id', 'gene', 'model'], as_index=False
).agg(
    median_delta=('delta', 'median'),
    median_absolute_delta=('absolute_delta', 'median'),
)

# Pool the patient-level medians for each alteration.
reliance_summary = reliance_patients.groupby(
    ['gene', 'model'], as_index=False
).agg(
    n_carriers=('record_id', 'nunique'),
    median_delta=('median_delta', 'median'),
    delta_q25=('median_delta', lambda x: x.quantile(0.25)),
    delta_q75=('median_delta', lambda x: x.quantile(0.75)),
    median_absolute_delta=('median_absolute_delta', 'median'),
)

# reliance_patients.to_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabpfn_model_reliance/reliance_patient_medians.csv', index=False)
# reliance_summary.to_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabpfn_model_reliance/reliance_summary.csv', index=False)